In [ ]:
import os, sys
import pandas as pd
from scipy.sparse import load_npz, save_npz, csr_matrix, triu
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import cooler
from schicluster.cool.utilities import get_chrom_offsets
import matplotlib
%matplotlib inline
import matplotlib.pylab as plt
import seaborn as sns
from scipy.stats import norm,zscore
from PyComplexHeatmap import *
import warnings
warnings.filterwarnings("ignore")
from PyComplexHeatmap import *
matplotlib.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rcParams['figure.dpi'] = 80
matplotlib.rcParams['savefig.dpi']=300

## Parameters

In [ ]:
workdir = "/home/x-wding2/project_wubin/BG/HIC/HBA_pseudobulk"
indir = None
res=10000
chrom_size_path = "/home/x-wding2/Ref/hg38/hg38_ucsc.nochrM.sizes"
metadata_path = "/home/x-wding2/Projects/BG/clustering/100kb/cell_metadata_with_annotation.tsv"
prefix='Subclass+Region' # used to define indir: {prefix}_compartment
loop_prefix=None #'Subclass' # Subclass+Region
group_col='Subclass' # Region # used to define figdir, and cell_types (group_order), match the col in metadata.columns
parent_col="Class" #used to sort the group_col (MajorType)
sort_cols=None
stratify_col = None #Subclass
stratify_figdir=None
palette_path = "/home/x-wding2/Projects/BG/metadata/HMBA_color_palette.xlsx"
split_order = ["Nonneuron", "Glut", "Splatter", "Subpallium GABA"]
figdir=None
cpu = 16
black_list_path=None
excluded_groups=['Low quality cells','doublet','Unk']

## region stratified by Subclass
group_col='Region'
loop_prefix='Subclass+Region'
stratify_col = 'Subclass'

In [ ]:
# Parameters
workdir = "/home/x-wding2/Projects/BG/HIC/BG_Class"
metadata_path = "/home/x-wding2/Projects/BG/clustering/100kb/annotations_without_Glut.tsv"
adata_prefix = "/home/x-wding2/Projects/BG/adata/BG"
atac_adata_path = "/home/x-wding2/Projects/BG/atac/HMBA.atac-5kb.h5ad"
atac_ann_path = "/home/x-wding2/Projects/BG/atac/obs.tsv"
pseudobulk_mcds_path = "/home/x-wding2/Projects/BG/pseudobulk/Class/Class.mcds"
pseudobulk_atac_adata_path = "/home/x-wding2/Projects/BG/atac/Pseudobulk.Class.5kb.h5ad"
chrom_size_path = "/home/x-wding2/Ref/hg38/hg38_ucsc.nochrM.sizes"
palette_path = "/home/x-wding2/Projects/BG/metadata/BG_color_palette.xlsx"
cg_count_path = "/home/x-wding2/Ref/hg38/annotations/CG-count-100k.tsv"
gene_annotation = "/home/x-wding2/Ref/hg38/annotations/gene_2000_2000.bed.gz"
marker_path = "/home/x-wding2/Projects/BG/metadata/BG_markers.xlsx"
marker_sheet = "BG"
mode = "raw"
prefix = "Class"
group_col = "Class"
parent_col = "Neighborhood"
sort_cols = ["Neighborhood"]
stratify_col = None
loop_prefix = None
query_str = None
excluded_groups = ["Low quality cells", "doublet", "Unk", "Glut"]
figdir = None
cpu = 8
indir = None
kernel_name = "m3c"
genome = "hg38"
obs_dim = "cell"
var_dim = "100kb"
var_dim5kb = "5kb"
extract_group = None
query_group = None
mc_types = ["CGN", "CHN"]
normalize_per_cell = True
split_order = ["Nonneuron", "Subpallium GABA", "Glut Sero Dopa", "Subpallium GABA-Glut"]
diff_comp_dir = None
dchic_env = "dchic"
param_yaml = "/home/x-wding2/Projects/Github/pym3c/pym3c/notebooks/templates/BG_Subclass_hic.yaml"
outdir = "/home/x-wding2/Projects/BG/HIC/BG_Class/loop"
outname = "1.Class.diff_loop.ipynb"


In [ ]:
if loop_prefix is None:
    loop_prefix=group_col
if not workdir is None:
    workdir=os.path.expanduser(workdir)
if not indir is None:
    indir=os.path.expanduser(indir)
else:
    assert not workdir is None
    indir=os.path.join(workdir,f"{loop_prefix}_loop")
if figdir is None:
    assert not workdir is None
    figdir=os.path.join(workdir,"loop",group_col)
else:
    figdir=os.path.expanduser(figdir)
if not os.path.exists(figdir):
    os.makedirs(figdir, exist_ok=True)
if black_list_path is None:
    black_list_path = os.path.join(workdir, f"{prefix}_raw/loop.blacklist.10kb.bed")
if loop_prefix is None:
    loop_prefix=group_col
chrom_size_path=os.path.expanduser(chrom_size_path)
metadata_path=os.path.expanduser(metadata_path)
chrom_size_path=os.path.expanduser(chrom_size_path)
print(indir,figdir,black_list_path)

In [ ]:
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
keep_chroms=[chrom for chrom in chrom_sizes.index if chrom not in ['chrX','chrY','chrM']]
chrom_sizes = chrom_sizes[keep_chroms]
df_chr=chrom_sizes.to_frame()
df_chr['N']=df_chr.index.to_series().apply(lambda x:int(x.lstrip('chr')) if x.lstrip('chr') not in ['X','Y'] else 30) #sort by chrom int (1,2,3..)
df_chr.sort_values('N',inplace=True)
chrom_sizes=chrom_sizes.loc[df_chr.index.tolist()]
if metadata_path.endswith('.csv') or metadata_path.endswith('.csv.gz'):
    metadata = pd.read_csv(metadata_path, index_col=0)
else:
    metadata = pd.read_csv(metadata_path, index_col=0,sep='\t')
# Filter out low quality cells
metadata=metadata.loc[~ (metadata[group_col].isin(excluded_groups))]
metadata=metadata.loc[metadata[group_col].apply(lambda x:not str(x).endswith('-Unk'))]
if not stratify_col is None:
    metadata=metadata.loc[~ (metadata[stratify_col].isin(excluded_groups))]
if not parent_col is None:
    metadata=metadata.loc[~ (metadata[parent_col].isin(excluded_groups))]
if sort_cols is None:
    sort_cols=parent_col
metadata.head()

In [ ]:
# define group_order (such as cell types) and stratify2group (such as Subclass to Subclass+Region)
if not sort_cols is None:
    metadata.sort_values(sort_cols,inplace=True)
else:
    metadata.sort_values(group_col,inplace=True)
sorted_groups=metadata[group_col].unique().tolist()
stratify2group={}
group_order=[]
if stratify_col is None:
    for g in sorted_groups:
        sname=g.replace(' ','_')
        infile=f'{indir}/{sname}/{sname}/{sname}.loop.bedpe'
        if os.path.exists(infile):
            with open(infile,'r') as f:
                N=len(f.readlines())
            if N>0:
                group_order.append(g)
else:
    if stratify_figdir is None:
        metadata['GroupID']=metadata[stratify_col].map(str)+'+'+metadata[group_col].map(str) #apply(lambda x:x.replace(' ','_')).
    else:
        metadata['GroupID']=metadata[group_col]
    stratify_values=metadata[stratify_col].unique().tolist()
    for s in stratify_values:
        s_name=s.replace(' ','_')
        if stratify_figdir is None:
            for g in sorted_groups:
                g_name=g.replace(' ','_')
                ct=f'{s_name}+{g_name}'
                if os.path.exists(os.path.join(indir,s_name,ct,f'{ct}.loop.bedpe')):
                    if s not in stratify2group:
                        stratify2group[s]=[]
                    stratify2group[s].append(g) # For example: keys are Subclass, values are Subclass+Region
                    group_order.append(f"{s}+{g}") # such as Subclass+Region: 'Microglia+CaB', 'Microglia+NAC','OPC+CaB'
        else:
            D=metadata.loc[:,[group_col,stratify_col]].drop_duplicates().groupby(stratify_col)[group_col].apply(lambda x:x.unique().tolist()).to_dict()
            for g in D[s]:
                g_name=g.replace(' ','_')
                if os.path.exists(os.path.join(indir,g_name,g_name,f'{g_name}.loop.bedpe')):
                    if s not in stratify2group:
                        stratify2group[s]=[]
                    stratify2group[s].append(g) # For example: keys are Subclass, values are Subclass+Region
                    group_order.append(f"{g}")

if not parent_col is None:
    parent2group=metadata.loc[:,[parent_col,group_col]].drop_duplicates().groupby(parent_col)[group_col].apply(lambda x:x.unique().tolist()).to_dict()
else:
    parent2group={}
parent2group['merged']=group_order
print(len(group_order),group_order)
print(stratify2group)
print(parent2group)

In [ ]:
# read color palette
D=pd.read_excel(os.path.expanduser(palette_path),
                sheet_name=None, index_col=0)
D1=D[group_col].Hex.to_dict()
group_colors={k: D1[k] for k in D1 if k in sorted_groups}
if not parent_col is None:
    parent_colors=D[parent_col].Hex.to_dict()
    if stratify_col is None:
        group2parent=metadata.loc[:,[group_col,parent_col]].drop_duplicates().set_index(group_col)[parent_col].to_dict()
    else:
        D1=D[stratify_col].Hex.to_dict()
        stratify_colors={k: D1[k] for k in D1 if k in stratify_values}
        group2parent=metadata.loc[:,[parent_col,'GroupID']].drop_duplicates().set_index('GroupID')[parent_col].to_dict()
    group2parent['merged']='merged'
else:
    parent_colors=None
    group2parent=None
print(parent_colors)
print(group_colors)
print(group2parent)

## ANOVA

ANOVA, or Analysis of Variance, is a statistical technique used to compare the means of two or more groups. It helps determine if there's a statistically significant difference between the average values (means) of these groups.</br>
Null Hypothesis (H0): This states that there is no significant difference between the means of the groups being compared.</br>
Alternative Hypothesis (H1): This states that there is at least one mean that is statistically different from the others.</br>

ANOVA calculates the following:</br>
Sum of squares: This measures the variability within and between groups, SSB = sum of squares between groups.</br>
Mean squares: This is the variance (average squared deviation from the mean) for each source of variation (between groups, within groups).</br>
F-statistic: This is a test statistic that compares the between-group variance to the within-group variance.</br>

组间均方（mean squared between, MSB），相当于每个族群相对于总体的方差；组内均方（mean squared error, MSE），也就是每个分布自身的方差 </br>
F=MSB / MSE
```
# SST = Total sum of squares = SSB + SSE
# SSB = sum of squares between groups
#
# SSE = sum of squares of errors
# MSB = SSB / (k-1); MSE = SSE / (N-k)
# F Value = f = MSB/MSE; The larger the F-statistic, the greater the variation between sample means relative to the variation within the samples.
# Thus, the larger the F-statistic, the greater the evidence that there is a difference between the group means.
# If the value of F is near about 1, then there is insignificant variance between the means of the two groups of data set under observation.
```

matrix: Q,E,T

    For each single cell, the imputed matrix of each chromosome Q was log-transformed, and Z -score normalized at each diagonal (result denoted as E, global background) and subtracted a local background between ≥ 30 kb and ≤ 50 kb (result denoted as T).

In [ ]:
def compute_anova(prefix_list, chrom, matrix='Q',outdir=None,chrom_sizes=None,res=None,
                  black_list_path=None):
    # if os.path.exists(f'{outdir}/{matrix}.F.{c}.npz'):
    #     return None
    n_bins = int(chrom_sizes.loc[chrom] // res) + 1 #how many bins for this chrom
    cov = np.zeros(n_bins)
    if not isinstance(black_list_path,list):
        black_list_path1=[black_list_path]
    else:
        black_list_path1=black_list_path
    # print(black_list_path)
    for blacklist in black_list_path1:
        bkl = pd.read_csv(os.path.expanduser(blacklist), sep='\t',
                          header=None, index_col=None)
        bkl_tmp = bkl.loc[(bkl[0] == chrom), [1, 2]].values // res
        for xx, yy in bkl_tmp:
            cov[xx - 7:yy + 7] = 1
    tot, last = 0, 0
    Esum, E2sum, Elast, E2last, ss_intra = [csr_matrix((n_bins, n_bins)) for i in range(5)] # all zeros
    for prefixes in prefix_list: #prefix_list is a list of list
        for prefix in prefixes:  #for ct in ctlist
            # cool_e = cooler.Cooler(f'{indir}/{ct}/{ct}/{ct}.{matrix}.cool')
            cool_e = cooler.Cooler(f'{prefix}.{matrix}.cool')
            E = triu(cool_e.matrix(balance=False, sparse=True).fetch(chrom))
            # cool_e2 = cooler.Cooler(f'{indir}/{ct}/{ct}/{ct}.{matrix}2.cool')
            cool_e2 = cooler.Cooler(f'{prefix}.{matrix}2.cool')
            E2 = triu(cool_e2.matrix(balance=False, sparse=True).fetch(chrom))
            n = cool_e.info['group_n_cells']
            Esum += E * n
            E2sum += E2 * n
            tot += n #total numbe of cells
            # print(c, ct)
        Egroup = Esum - Elast
        E2group = E2sum - E2last
        Egroup.data = Egroup.data ** 2 / (tot - last)
        ss_intra += (E2group - Egroup)
        Elast = Esum.copy()
        E2last = E2sum.copy()
        last = tot
    Esum.data = Esum.data ** 2 / tot
    ss_total = E2sum - Esum
    ss_intra.data = 1 / ss_intra.data
    ss_total = ss_total.multiply(ss_intra)
    # print(c, ss_total.data.min(), ss_intra.data.min())

    ss_total.data = (ss_total.data - 1) * (tot - len(prefix_list)) / (len(prefix_list) - 1)
    ss_total = ss_total.tocoo()
    bklfilter = np.logical_and(cov[ss_total.row]==0, cov[ss_total.col]==0)
    distfilter = np.logical_and((ss_total.col-ss_total.row)>5, (ss_total.col-ss_total.row)<500)
    idxfilter = np.logical_and(bklfilter, distfilter)
    # print(idxfilter.sum(), len(idxfilter))
    ss_total = csr_matrix((ss_total.data[idxfilter], (ss_total.row[idxfilter], ss_total.col[idxfilter])), (n_bins, n_bins))
    save_npz(f'{outdir}/{matrix}.F.{chrom}.npz', ss_total) # F statistic, old name: {matrix}pv_{chrom}.npz
    return [chrom, matrix, tot]

def chrom_iterator(indir, chrom_order, matrix, chrom_offset):
    for chrom in chrom_order:
        infile = f'{indir}/{matrix}.F.{chrom}.npz'
        if not os.path.exists(infile):
            continue
        chunk_size = 5000000
        data = load_npz(infile).tocoo()
        df = pd.DataFrame({'bin1_id': data.row, 'bin2_id': data.col, 'count': data.data})
        df = df[df['bin1_id'] <= df['bin2_id']]
        for i, chunk_start in enumerate(range(0, df.shape[0], chunk_size)):
            chunk = df.iloc[chunk_start:chunk_start + chunk_size]
            # chunk.iloc[:, :2] += chrom_offset[chrom]
            cols=chunk.columns.tolist()
            for col in cols[:2]:
                chunk.loc[:,col]+=chrom_offset[chrom]
            yield chunk

In [ ]:
bins_df = cooler.binnify(chrom_sizes, res)
chrom_offset = get_chrom_offsets(bins_df)
print(bins_df.head())
print(chrom_offset)

In [ ]:
# diff loop between_groups (parent_col if stratify_col is None) or between stratify
prefix_list=[]
if stratify_col is None:
    if not parent_col is None:
        for parent in parent2group: # NonN: [ODC, OPC],..; or Microglia: [CaB, NAC]
            if parent=='merged':
                continue
            prefix=[]
            for ct in parent2group[parent]:
                sname=ct.replace(' ','_')
                path=f'{indir}/{sname}/{sname}/{sname}'
                if ct not in group_order:
                    continue
                print(path)
                if os.path.exists(f"{path}.Q.cool"):
                    prefix.append(path)
            if len(prefix)>0:
                prefix_list.append(prefix)
        print(prefix_list,black_list_path)
else:
    for stratify in stratify2group: # NonN: [ODC, OPC],..; or Microglia: [CaB, NAC]
        prefix=[]
        stratify_name=stratify.replace(' ','_')
        for g2 in stratify2group[stratify]:
            sname=g2.replace(' ','_')
            ct=f"{stratify}+{g2}"
            ct_name=ct.replace(' ','_')
            if ct not in group_order:
                continue
            path=f'{indir}/{stratify_name}/{ct_name}/{ct_name}'
            # print(path)
            if os.path.exists(f"{path}.Q.cool"):
                prefix.append(path)
        if len(prefix)>0:
            prefix_list.append(prefix)
    print(prefix_list,black_list_path)
if len(prefix_list)>0:
    bname=parent_col if stratify_col is None else stratify_col
    with ProcessPoolExecutor(cpu) as executor:
        futures = []
        for chrom in chrom_sizes.index:
            for matrix in ['Q', 'E', 'T']:
                if os.path.exists(os.path.join(figdir,f'{bname}.{matrix}.anova.cool')):
                    print("existed:",os.path.join(figdir,f'{bname}.{matrix}.anova.cool'))
                    continue
                future = executor.submit(compute_anova, prefix_list=prefix_list, chrom=chrom,
                                         matrix=matrix, outdir=figdir,
                                         chrom_sizes=chrom_sizes, res=res, black_list_path=black_list_path)
                futures.append(future)
        for future in as_completed(futures):
            tmp = future.result()
            if not tmp is None:
                print(f'{tmp[0]} {tmp[1]}',end=',')
    for matrix in ['Q', 'E', 'T']: # merge chroms .npz into one cool file
        output_path = os.path.join(figdir,f'{bname}.{matrix}.anova.cool') #merge {matrix}.F.{chrom}.cool into {matrix}.cool
        if os.path.exists(output_path):
            print(f"{output_path} existed")
            continue
        cooler.create_cooler(cool_uri=output_path,bins=bins_df,
                             pixels=chrom_iterator(figdir, chrom_sizes.index, matrix, chrom_offset),
                             ordered=True,dtypes={'count': np.float32})
    os.system(f"rm -f {figdir}/*.F.*.npz")

In [ ]:
# one vs rest (all cell types)
if stratify_col is None:
    prefix_list=[]
    for ct in group_order:
        sname=ct.replace(' ','_')
        prefix_list.append([f'{indir}/{sname}/{sname}/{sname}'])
    print(prefix_list,black_list_path) #each MajorType (groupby) is a separated group
    with ProcessPoolExecutor(cpu) as executor:
        futures = []
        for chrom in chrom_sizes.index:
            for matrix in ['Q', 'E', 'T']:
                if os.path.exists(os.path.join(figdir,f'{group_col}.{matrix}.anova.cool')):
                    continue
                future = executor.submit(compute_anova, prefix_list=prefix_list, chrom=chrom,
                                         matrix=matrix, outdir=figdir,
                                         chrom_sizes=chrom_sizes, res=res, black_list_path=black_list_path)
                futures.append(future)
        for future in as_completed(futures):
            tmp = future.result()
            if not tmp is None:
                print(f'{tmp[0]} {tmp[1]}',end=',')
    for matrix in ['Q', 'E', 'T']: # merge chroms .npz into one cool file
        output_path = os.path.join(figdir,f'{group_col}.{matrix}.anova.cool') #merge {matrix}.F.{chrom}.cool into {matrix}.cool
        if os.path.exists(output_path):
            continue
        cooler.create_cooler(cool_uri=output_path,bins=bins_df,
                             pixels=chrom_iterator(figdir, chrom_sizes.index, matrix, chrom_offset),
                             ordered=True,dtypes={'count': np.float32})
    os.system(f"rm -f {figdir}/*.F.*.npz")

## Merge diff loop's Q

In [ ]:
# Load Loop Position and ANOVA F statistic
# bedpe_cols = ['chrom', 'x1', 'x2', 'chrom', 'y1', 'y2', 'E']
if not os.path.exists(os.path.join(figdir,'merged_loop_with_anova.tsv')):
    df_loop=[]
    for ct in group_order:
        sname=ct.replace(' ','_')
        if stratify_col is None:
            infile=f'{indir}/{sname}/{sname}/{sname}.loop.bedpe'
        else:
            stratify_name=ct.split('+')[0].replace(' ','_')
            infile=f'{indir}/{stratify_name}/{sname}/{sname}.loop.bedpe'
        if not os.path.exists(infile):
            continue
        with open(infile,'r') as f:
            N=len(f.readlines())
        if N==0:
            continue
        df1=pd.read_csv(infile,sep='\t',index_col=None, header=None)
        df_loop.append(df1)
    df_loop = pd.concat(df_loop, axis=0)
    print(df_loop.head())
    df_loop = df_loop.drop([6], axis=1).drop_duplicates(subset=[0, 1, 4]).sort_values([0, 1, 4])
    df_loop = pd.concat([df_loop[(df_loop[0] == c).values] for c in chrom_sizes.index])
    df_loop.index = np.arange(df_loop.shape[0])
    # df_loop: chr1,start1,end1,chr2,start2,end2
    df_loop.columns=['chr1','start1','end1','chr2','start2','end2']
    # df_loop.to_csv(os.path.join(figdir,'merged_loop.bedpe'),sep='\t', index=False, header=False)
    
    # add diff loop anova into df_loop
    for chrom in chrom_sizes.index:
        loopfilter = (df_loop['chr1'] == chrom)
        looptmp = df_loop.loc[loopfilter, ['start1', 'start2']].values // res # start position to bin IDs
        for matrix in ['Q', 'E', 'T']:
            # between group (CellClass) diff loop
            if stratify_col is None and not parent_col is None:
                cool_path=os.path.join(figdir, f'{parent_col}.{matrix}.anova.cool')
                if os.path.exists(cool_path):
                    cool = cooler.Cooler(cool_path)
                    f_values = triu(cool.matrix(balance=False, sparse=True).fetch(chrom)).tocsr() # read anova F statistic
                    df_loop.loc[loopfilter, f'{parent_col}.{matrix}.F'] = f_values[(looptmp[:, 0], looptmp[:, 1])].A1
            elif not stratify_col is None:
                cool_path=os.path.join(figdir, f'{stratify_col}.{matrix}.anova.cool')
                if os.path.exists(cool_path):
                    cool = cooler.Cooler(cool_path)
                    f_values = triu(cool.matrix(balance=False, sparse=True).fetch(chrom)).tocsr() # read anova F statistic
                    df_loop.loc[loopfilter, f'{stratify_col}.{matrix}.F'] = f_values[(looptmp[:, 0], looptmp[:, 1])].A1 # subset the corresponding bins
            if stratify_col is None:
                cool_path=os.path.join(figdir, f'{group_col}.{matrix}.anova.cool')
                if os.path.exists(cool_path):
                    cool = cooler.Cooler(cool_path)
                    f_values = triu(cool.matrix(balance=False, sparse=True).fetch(chrom)).tocsr() # read anova F statistic
                    df_loop.loc[loopfilter, f'{group_col}.{matrix}.F'] = f_values[(looptmp[:, 0], looptmp[:, 1])].A1 # subset the corresponding bins
    print(df_loop.columns.values,df_loop.shape)
    df_loop.to_csv(os.path.join(figdir,'merged_loop_with_anova.tsv'),sep='\t',index=False) #included anvoa F statistic
else:
    df_loop=pd.read_csv(os.path.join(figdir,'merged_loop_with_anova.tsv'),sep='\t')
df_loop

In [ ]:
def load_Q(ct, m, df_loop, indir, chrom_sizes, res):
    """
    Load Q and T matrix and subset the diff loop bins 
    """
    tmp = []
    sname=ct.replace(' ','_')
    if '+' in sname:
        group=sname.split('+')[0]
    else:
        group=sname
    cool_file = cooler.Cooler(os.path.join(indir,f'{group}/{sname}/{sname}.{m}.cool')).matrix(balance=False, sparse=True)
    for chrom in chrom_sizes.index:
        if chrom not in df_loop['chr1'].unique().tolist():
            continue
        mat = cool_file.fetch(chrom).tocsr()
        tmp.append(mat[(df_loop.loc[df_loop['chr1']==chrom, 'start1'].values // res, df_loop.loc[df_loop['chr1']==chrom, 'start2'].values // res)].A1)
    return [ct, np.concatenate(tmp)]

In [ ]:
# Load Q and T of the diff loops
for matrix in ['Q','T']:
    if os.path.exists(os.path.join(figdir,f'merged_loop_{matrix}.tsv')):
        continue
    with ProcessPoolExecutor(cpu) as executor:
        futures = []
        for ct in group_order:
            future = executor.submit(load_Q,
                ct=ct,m=matrix,df_loop=df_loop,indir=indir,chrom_sizes=chrom_sizes,res=res,
            )
            futures.append(future)

        loop_matrix = []
        for future in as_completed(futures):
            tmp = future.result()
            loop_matrix.append(pd.DataFrame(tmp[1], columns=[tmp[0]]))
            print(f'{tmp[0]}',end=',')
    loop_matrix = pd.concat(loop_matrix, axis=1) #rows are loop bins, columns are cell types
    loop_matrix = loop_matrix[group_order]
    #loop_matrix has the same index and row order as df_loop
    print(loop_matrix.columns.values,loop_matrix.shape)
    loop_matrix.to_csv(os.path.join(figdir,f'merged_loop_{matrix}.tsv'), sep='\t',index=False)
    # norm.isf(x) = norm.ppf(1-x)

## Plot Diff Loops

In [ ]:
thres1 = norm.isf(0.025) #97.5% percentile
thres2 = norm.isf(0.15) #same as norm.ppf(0.85), ppf: percent-point function = the inverse of the cumulative distribution function (CDF); P(Z≤z)=0.85
print(thres1, thres2)

In [ ]:
if not stratify_col is None:
    cols=[stratify_col]
else:
    cols=[group_col] if parent_col is None else [group_col,parent_col]
matrix=['Q','T']
width=max(5,len(cols)*2*3.1)
fig, axes = plt.subplots(2, len(cols)*2, figsize=(width, 6.2), dpi=300)
print(width,axes.shape)
i=0
for m in matrix:
    j=0
    for col in cols:
        col_name=f'{col}.{m}.F'
        ax = axes[i,j]
        sns.histplot(zscore(np.log10(df_loop[col_name]+0.01)), bins=100, binrange=(-4,4), ax=ax,color='dodgerblue') #log_scale=(None,10)
        # ax2 = ax.twinx()
        # ax2=sns.kdeplot(zscore(np.log10(df_loop[col_name])),ax=ax2,color='red')
        ax.set_xlabel(f"zscore log10({col} {m} anova F ",fontsize=10)
        ax = axes[i,j+1]
        sns.histplot(zscore(df_loop[col_name]), bins=100, binrange=(-2,4), ax=ax,color='dodgerblue')
        ax.plot([thres1,thres1],[0,1],transform=ax.get_xaxis_transform(),color='green',linestyle='--')
        ax.plot([thres2,thres2],[0,1],transform=ax.get_xaxis_transform(),color='red',linestyle='--')
        # ax2 = ax.twinx()
        # ax2=sns.kdeplot(zscore(df_loop[col_name]),ax=ax2,color='red')
        ax.set_xlabel(f"zscore {col} {m} anova F ",fontsize=10)
        plt.tight_layout()
        j+=2
    i+=1
plt.savefig(os.path.join(figdir,'anova_f_statistic_distplot.pdf'), transparent=True,bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
loopq = pd.read_csv(os.path.join(figdir,f'merged_loop_Q.tsv'),sep='\t')
loopt = pd.read_csv(os.path.join(figdir,f'merged_loop_T.tsv'),sep='\t')

In [ ]:
loopq.head(),loopt.head()

In [ ]:
group_name=group_col if stratify_col is None else "GroupID"
df_cols=loopq.columns.to_frame(name=group_name)
if not stratify_col is None:
    df_cols[stratify_col]=df_cols.GroupID.apply(lambda x:x.split('+')[0])
    df_cols[group_col]=df_cols.GroupID.apply(lambda x:x.split('+')[1])
if not parent_col is None:
    df_cols[parent_col]=df_cols[group_name].map(group2parent)
    df_cols.sort_values([parent_col,group_name],inplace=True)
else:
    if split_order is None:
        df_cols.sort_values(group_name,inplace=True)
    else:
        df_cols=df_cols.loc[split_order]
df_cols

In [ ]:
cols=loopq.columns.tolist()
max_idx=np.argmax(loopq.values,axis=1)
df_loop['Group']=[cols[i] for i in max_idx]
df_loop

In [ ]:
from PyComplexHeatmap.utils import despine
if stratify_col is None:
    cols=[group_col] if parent_col is None else [group_col,parent_col]
else:
    cols=[stratify_col]
for col in cols:
    print(col)
    # select the significant differential loop
    sig_loops = ((zscore(df_loop[f'{col}.Q.F'])>thres2) & (zscore(df_loop[f'{col}.T.F'])>thres2))
    print(sig_loops.sum())
    
    df_rows=df_loop.loc[sig_loops]
    if not stratify_col is None:
        df_rows[stratify_col]=df_rows.Group.apply(lambda x:x.split('+')[0])
        df_rows[group_col]=df_rows.Group.apply(lambda x:x.split('+')[1])
    if not parent_col is None:
        df_rows[parent_col]=df_rows.Group.map(group2parent)
    print(df_rows.head())
    row_order=[]
    if stratify_col is None:
        for g in df_cols[group_col].tolist():
            rows=df_rows.loc[df_rows.Group==g].sort_values(f'{col}.Q.F',ascending=False).index.tolist()
            row_order.extend(rows)
    else:
        parent2stratify=df_rows.loc[:,[parent_col,stratify_col]].drop_duplicates().groupby(parent_col)[stratify_col].apply(lambda x:x.unique().tolist()).to_dict()
        for g2 in df_cols['GroupID'].unique():
            rows=df_rows.loc[(df_rows['Group']==g2)].sort_values(f'{stratify_col}.Q.F',ascending=False).index.tolist()
            row_order.extend(rows)
    df_rows=df_rows.loc[row_order]
    tmpq = loopq.loc[df_rows.index.tolist(),df_cols.index.tolist()]
    tmpt = loopt.loc[df_rows.index.tolist(),df_cols.index.tolist()]
    
    print(tmpq.shape,tmpt.shape)
    if not parent_col is None:
        tmp_colors={}
        for p in df_rows[parent_col].unique():
            tmp_colors[p]=parent_colors[p]
        row_ha_dict={parent_col:anno_simple(df_rows[parent_col],colors=tmp_colors,legend=False)}
        row_ha = HeatmapAnnotation(**row_ha_dict,axis=0,
                 verbose=0,label_kws=dict(visible=False))
        row_split=df_rows[parent_col]
        row_split_order=df_cols[parent_col].unique().tolist() if split_order is None else [o for o in split_order if o in df_cols[parent_col].tolist()]

        # col_ha
        tmp_colors={}
        for p in df_cols[parent_col].unique():
            tmp_colors[p]=parent_colors[p]
        col_ha_dict={
                    parent_col:anno_simple(
                        df_cols[parent_col],height=5,
                         colors=tmp_colors,
                         # add_text=True,text_kws={'fontweight':'bold','color':'white'},
                        legend=True),
                    }
        col_ha = HeatmapAnnotation(**col_ha_dict,axis=1,
                 verbose=0,label_kws=dict(visible=False))
        col_split=df_cols[parent_col]
        col_split_order=df_cols[parent_col].unique().tolist() if split_order is None else [o for o in split_order if o in df_cols[parent_col].tolist()]
    else:
        row_ha=None
        row_split=None
        row_split_order=None
        col_ha=None
        col_split=None
        col_split_order=None

    cm1 = ClusterMapPlotter(
                data=tmpq, top_annotation=col_ha, left_annotation=row_ha,
                show_rownames=False, show_colnames=True,col_cluster=False,row_cluster=False,
                row_dendrogram=False, col_dendrogram=False,
                row_cluster_method='ward',row_cluster_metric='cosine',
                col_split=col_split, col_split_order=col_split_order,
                xticklabels_kws={'labelrotation':-60,'labelcolor':'black','labelsize':10},xticklabels=True,
                col_split_gap=0.4,cmap='BlueDarkRed18',#turbo, parula, viridis,RedYellowBlue_r,BlueOrangeRed,
                # row_split=df_rows.Group,row_split_order=df_cols.MajorType.tolist(),
                row_split=row_split,row_split_gap=0.8,row_split_order=row_split_order,
                rasterized=True, #row_split_gap=0.1,vmax=1,vmin=0,center=0.5,
                plot=False,label='Normalized Interaction',
                standard_scale=0,
                # z_score=0,
                verbose=1)

    cm2 = ClusterMapPlotter(
                data=tmpt, top_annotation=col_ha, #left_annotation=row_ha,
                show_rownames=False, show_colnames=True,col_cluster=False,row_cluster=False,
                row_dendrogram=False, col_dendrogram=False,xticklabels=True,
                col_split=col_split, col_split_order=col_split_order,
                xticklabels_kws={'labelrotation':-60,'labelcolor':'black','labelsize':10},
                col_split_gap=0.4,row_split_gap=0.8,rasterized=True,
                plot=False,cmap='BlueDarkRed18',label='Normalized Interaction',
                standard_scale=0,
                # z_score=0,
                verbose=0)
    cmlist=[cm1,cm2]
    nrows,ncols=tmpq.shape
    width=max(ncols*0.5,5)
    plt.figure(figsize=(width,10))
    ax,legend_axes=composite(cmlist=cmlist, main=0,legend_hpad=2,col_gap=0.1,legend_gap=10)
    cm1.ax.set_title("Q",fontsize=14)
    cm2.ax.set_title("T",fontsize=14)
    ax.set_ylabel(str(tmpq.shape[0])+" Diff Loop",color='black',fontsize=14)
    ax.set_axis_on()
    despine(ax=ax, left=True, bottom=True,top=True,right=True)
    ax.tick_params(axis="both",which="both",left=False,
                    right=False,labelleft=False,labelright=False,
                    top=False,bottom=False,labeltop=False,labelbottom=False)
    ax.set_title(f"Diff Loop between {col}",y=1.03,fontdict={'fontweight':'bold','color':'blue'},fontsize=14)
    plt.savefig(os.path.join(figdir,f'{group_col}.diff_loop_Q_T_between_{col}_heatmap.pdf'), transparent=True,bbox_inches='tight')
    plt.show()
    print(df_rows['Group'].value_counts().to_dict())
    if not parent_col is None:
        print(df_rows[parent_col].value_counts().to_dict())

In [ ]:
if not stratify_col is None:
    df=df_rows.Group.value_counts().to_frame()
    df[stratify_col]=df.index.to_series().apply(lambda x:x.split('+')[0])
    df[group_col]=df.index.to_series().apply(lambda x:x.split('+')[1])
    df=df.set_index([stratify_col,group_col]).unstack().fillna(0)
    df=df.apply(lambda x:x/x.sum(),axis=1)['count']

    stratify_colors1={}
    for p in df.index.unique():
        stratify_colors1[p]=stratify_colors[p]
    plt.figure
    ax=df.loc[list(stratify_colors1.keys())].plot(kind='bar',stacked=True,color=group_colors,align='center', width=0.9)
    ax.legend(loc='upper left',frameon=True,bbox_to_anchor=(1.0,1))
    ax.set_ylim(0,1)
    sns.despine(ax=ax,top=True,right=True)
    plt.savefig(os.path.join(figdir,'percentage_of_diff_loop.barplot.pdf'),transparent=True, bbox_inches='tight',dpi=300)
    plt.show()

    group_colors1={}
    for p in df.columns.unique():
        group_colors1[p]=group_colors[p]
    plt.figure(figsize=(4,3))
    ax=sns.countplot(data=df_rows,x=group_col,palette=group_colors1,order=list(group_colors1.keys()))
    # ax.set_xticklabels(hue_order, rotation=90)
    ax.set_title("No. of differential loop",fontsize=16)
    plt.savefig(os.path.join(figdir,'Number_of_diff_loop.barplot.pdf'),transparent=True, bbox_inches='tight',dpi=300)
    plt.show()

## Aggregate peak analysis of differential loops

Aggregate Peak Analysis (APA): https://github.com/aidenlab/juicer/wiki/APA.
	It can be very informative to view the average Hi-C matrix for a set of regions, rather than the Hi-C matrix at each individual region. This can help you in identifying common structural features of these regions or over-/under-representation of contacts in the vicinity.

In [ ]:
df_loop=pd.read_csv(os.path.join(figdir,'merged_loop_with_anova.tsv'),sep='\t')
df_loop

In [ ]:
loopq = pd.read_csv(os.path.join(figdir,f'merged_loop_Q.tsv'),sep='\t')
loopt = pd.read_csv(os.path.join(figdir,f'merged_loop_T.tsv'),sep='\t')
loopq.head(),loopt.head()

In [ ]:
thres1 = norm.isf(0.025)
thres2 = norm.isf(0.15)
print(thres1, thres2)

In [ ]:
loopq

In [ ]:
def Apa(ct,loop_list,res,loop_dir,chrom_sizes,dist = 505):
    sname=ct.replace(' ','_')
    if '+' in sname:
        group=sname.split('+')[0]
    else:
        group=sname
    result = np.zeros((len(loop_list), 21, 21))
    cool = cooler.Cooler(os.path.join(loop_dir,f"{group}/{sname}/{sname}.Q.cool")) #load imputed Q matrix
    for c in np.random.permutation(chrom_sizes.index):
        # print(ct, c,"\t"*6,end='\r')
        Q = cool.matrix(balance=False, sparse=True).fetch(c).toarray()
        decay = np.array([np.mean(np.diag(Q, k)) for k in range(dist)]) # diagonal: average Q of all interaction between bin(i) and bin(i) and bin(i-k)
        E = np.zeros(Q.shape)
        row, col = np.diag_indices(E.shape[0])
        E[row, col] = 1
        for k in range(1, dist):
            # the imputed matrix of each chromosome Q was log-transformed, and Z -score normalized at each diagonal (result denoted as E, global background)
            # and subtracted a local background between ≥ 30 kb and ≤ 50 kb (result denoted as T).
            E[row[:-k], col[k:]] = (Q[row[:-k], col[k:]]) / decay[k] #normalized by the mean of matrix's diagonal
        j=0
        for j, loop in enumerate(loop_list): # diff loop filtered based on fold change and distance
            looptmp = loop.loc[(loop['chr1'] == c), ['start1', 'start2']].values // res # bin ID for start1 and start2
            for xx, yy in looptmp: #for the loop (bin i and bin j interaction), get the z_scored values around nearby 21 bins window matrix centered at i,j.
                tmp = E[(xx - 10):(xx + 11), (yy - 10):(yy + 11)] # window size = 20: upsteram and downstream 10 bins
                tmp = (tmp - tmp.min()) / (tmp.max() - tmp.min()) # z score normalized, similar to max pool
                result[j] += tmp #sum up the zscored values (21*21) for all loop of the given cell type
    for j, loop in enumerate(loop_list):
        result[j] /= loop.shape[0]
    return result #resulting 21*21 matrix, representing the average E near each loop

In [ ]:
usecol=group_col if stratify_col is None else stratify_col
if not os.path.exists(os.path.join(figdir,f'{usecol}_diffloop_apa.npz')):
    # filter diff loops based on ANOVA F statistic
    sig_loops = ((zscore(df_loop[f'{usecol}.Q.F'])>thres2) & (zscore(df_loop[f'{usecol}.T.F'])>thres2))
    # filter diff loops based on loop distance and distance to telomere
    distfilter = ((df_loop['start2'] - df_loop['start1']) > 100000) & ((df_loop['start2'] - df_loop['start1']) < 4950000) & (
        df_loop['start1'] >= (10 * res)) & (df_loop['start2'] < (df_loop['chr1'].map(chrom_sizes) - 10 * res))
    loopfilter = distfilter & sig_loops
    print(sig_loops.sum(),distfilter.sum(), loopfilter.sum())
    df_loop = df_loop.loc[loopfilter]
    loopq = loopq.loc[loopfilter]
    loopt = loopt.loc[loopfilter]

    fold_change_q,fold_change_t=1.2,1.5
    dist=505

    loop_list = [] #store the diff loops filtered by fold change of Q and T for each cell type, has the same order as cell_types
    for ct in group_order:
        ctlist=[ct]
        fc1 = loopq[ctlist].mean(axis=1) / loopq.loc[:, ~loopq.columns.isin(ctlist)].mean(axis=1) #Q: imputed matrix
        # #Q was log-transformed, and Z -score normalized at each diagonal (result denoted as E, global background) and subtracted a local background between ≥ 30 kb and ≤ 50 kb (result denoted as T)
        fc2 = loopt[ctlist].mean(axis=1) / loopt.loc[:, ~loopt.columns.isin(ctlist)].mean(axis=1)
        loop_list.append(df_loop.loc[(fc1 > fold_change_q) & (fc2 > fold_change_t)])
    print(len(loop_list))

    result = np.zeros((len(group_order), len(loop_list), 21, 21))
    with ProcessPoolExecutor(cpu) as executor:
        futures = {}
        for i, ct in enumerate(group_order):
            future = executor.submit(Apa,
                ct=ct,loop_list=loop_list,res=res,loop_dir=indir,
                chrom_sizes=chrom_sizes,dist = dist
            )
            futures[future] = i

        for future in as_completed(futures):
            idx = futures[future]
            result[idx] = future.result()
            print(f'{group_order[idx]}',end=',')
    print(result.shape)
    np.savez(os.path.join(figdir,f'{usecol}_diffloop_apa.npz'),data=dict(result=result, group_order=group_order,loop_list=loop_list))
    del loop_list
else:
    data=np.load(os.path.join(figdir,f'{usecol}_diffloop_apa.npz'),allow_pickle=True)['data'].ravel()[0]
    result,group_order=data['result'],data['group_order']
    del data

In [ ]:
def plot_apa(celltypes,group_order,result,stratify_col,group_colors,parent_colors,group2parent,bname):
    nrows=len(celltypes)
    width=max(3.5,nrows)
    fontsize1=10
    fontsize2=6
    rotation=None
    if width > 18:
        width=18
        fontsize1=8
        fontsize2=4
    rotation=45
    fig, axes = plt.subplots(nrows, nrows, figsize=(width, width),
                             sharex='all', sharey='all', dpi=300,
                             gridspec_kw=dict(hspace=0.05,wspace=0.05)
                            )
    fig.subplots_adjust(wspace=0.2, hspace=0.6)
    for i,ct1 in enumerate(celltypes):
        row_idx=group_order.index(ct1)
        for j,ct2 in enumerate(celltypes): #columns, diff loop position
            col_idx=group_order.index(ct2)
            # print(ct1,ct2,row_idx,col_idx)
            if isinstance(axes,np.ndarray):
                ax = axes[i, j]
            else:
                ax=axes
            tmp = result[row_idx,col_idx]
            apa = np.around(tmp[10, 10] / tmp[-5:, :5].mean(), decimals=2) # E of the loop (i,j) = E(10,10), divided by the mean of lower left part of 5*5 window's matrix
            ax.imshow(tmp, cmap='hot_r', vmax=0.8, vmin=0.2)
            ax.tick_params(axis="x",which="both",top=False,bottom=False,labeltop=False,labelbottom=False)
            if j > 0:
                ax.tick_params(axis="y",which="both",left=False,right=False,labelleft=False,labelright=False)
            if i == 0:
                if not stratify_col is None:
                    title=ct2.split('+')[1]
                    color=group_colors[title]
                else:
                    title=ct2
                    color=parent_colors[group2parent[title]] if not parent_col is None else 'black'
                ax.set_title(f'{title}', fontsize=fontsize1,color=color,rotation=rotation)
            ax.text(0.05, 0.05, apa, ha='left', va='bottom', fontsize=8,transform=ax.transAxes) #upper left / upper right

    for i, ct in enumerate(celltypes):
        if isinstance(axes,np.ndarray):
            ax=axes[i, 0]
        else:
            ax=axes
        if not stratify_col is None:
            ylabel=ct.split('+')[1]
            color=group_colors[ylabel]
        else:
            color=parent_colors[group2parent[ct]] if not parent_col is None else 'black'
            ylabel=ct
        ax.set_ylabel(ylabel,fontsize=fontsize1,labelpad=-5,color=color,rotation=rotation,horizontalalignment='right')
        ax.set_yticks([0, 20])
        ax.grid(False)
        ax.set_yticklabels(['-100k', '+100k'],fontsize=fontsize2)
    # ax.set_xticks([0, 20]) #21*21 matrix, 20 * 10kb = 200kb, centered at loop (i,j)
    # ax.set_xticklabels(['-100k', '+100k'])
    plt.tight_layout()
    outname=os.path.join(figdir,f'{bname}_diffloop_apa.pdf')
    plt.savefig(outname,transparent=True,bbox_inches='tight',dpi=300)

In [ ]:
if stratify_col is None:
    plot_apa(group_order,group_order,result,stratify_col,group_colors,parent_colors,group2parent,group_col)
else:
    for stratify in stratify2group:
        celltypes=[f"{stratify}+{ct}" for ct in sorted(stratify2group[stratify]) if f"{stratify}+{ct}" in group_order]
        plot_apa(celltypes,group_order,result,stratify_col,group_colors,parent_colors,group2parent,stratify.replace(' ','_'))

Each of these submatrices is a 210 kb x 210 kb square centered at a single putative peak in the upper triangle of the contact matrix. The resulting APA plot displays the total number of contacts that lie within the entire putative peak set at the center of the matrix; the entry immediately to the right of center corresponds to the total number of contacts in the pixel set obtained by shifting the peak set 10 kb to the right; the entry two positions above center corresponds to an upward shift of 20 kb and so on. Focal enrichment across the peak set in aggregate manifests as larger values at the center of the APA plot.

Average interaction between each loop and nearby bins.
For each row (cell type), load the normalized E matrix for this cell type, calculate the APA (21*21) at the center of diff loop for each cell type (column).</br>
rows: E</br>
Columns: diff loop position

## (Revision) Cross-cell-type APA strength matrix

Same analysis as in `1.Subclass.diff_loop.ipynb`, applied at the Class level.
For every (loop-set Class *i*, signal Class *j*) we summarise the 21×21 APA
submatrix as a single *APA strength* = central-pixel value / mean of the corner
background. The resulting N×N matrix is clustered to reveal a "loop-space
lineage tree" — Classes that share chromatin-loop programmes group together,
even when the number of called loops differs.


In [ ]:
# Reload the per-celltype APA tensor saved by the existing pipeline.
# `result[i, j]` is the 21x21 APA submatrix obtained by piling up
# *cell-type i*-specific diff loops on the imputed contact matrix of cell type j.
import os, numpy as np, pandas as pd
from PyComplexHeatmap import ClusterMapPlotter, HeatmapAnnotation, anno_simple
import matplotlib.pyplot as plt

usecol = group_col if stratify_col is None else stratify_col
apa_npz = os.path.join(figdir, f'{usecol}_diffloop_apa.npz')
data = np.load(apa_npz, allow_pickle=True)['data'].ravel()[0]
result      = data['result']                # (N, N, 21, 21)
group_order = list(data['group_order'])     # length N
print('APA tensor shape:', result.shape)

def apa_strength(mat):
    '''Center pixel divided by mean of the lower-left 5x5 corner (background).'''
    center = mat[10, 10]
    bg     = mat[-5:, :5].mean()
    return center / bg if bg > 0 else np.nan

# Build NxN APA strength matrix.
N = len(group_order)
apa_mat = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        apa_mat[i, j] = apa_strength(result[i, j])
df_apa = pd.DataFrame(apa_mat, index=group_order, columns=group_order)
df_apa.index.name   = 'signal_celltype'
df_apa.columns.name = 'loop_set_celltype'
df_apa.to_csv(os.path.join(figdir, f'{usecol}.cross_celltype_apa_strength.tsv'),
              sep='\t')
df_apa.head()


In [ ]:
# Clustered heatmap. Rows (i) = cell type providing the signal Q matrix;
# columns (j) = cell type providing the diff-loop set that is piled up on
# row i's Q matrix. Diagonal (i==j) is self-recovery and should be highest;
# off-diagonal blocks reveal lineage relationships in loop space.
# To enforce a *diagonal* arrangement (row and column order identical), we
# pre-compute a single hierarchical ordering on the symmetrised matrix and
# disable independent row/col clustering inside ClusterMapPlotter.
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
sym = (df_apa.values + df_apa.values.T) / 2.0
np.fill_diagonal(sym, sym.max())
dist = sym.max() - sym
np.fill_diagonal(dist, 0.0)
dist = (dist + dist.T) / 2.0  # ensure perfect symmetry
if N > 2:
    Z = linkage(squareform(dist, checks=False), method='average')
    diag_order = df_apa.index[leaves_list(Z)].tolist()
else:
    diag_order = df_apa.index.tolist()
df_apa = df_apa.loc[diag_order, diag_order]
apa_mat = df_apa.values  # refresh for the summary block below

# Per-Class cell counts, appended to the right-hand row labels as "Class (n)".
_ann = pd.read_csv('/anvil/projects/x-mcb130189/Wubin/BG/clustering/100kb/annotations.tsv',
                   sep='\t', index_col=0)
class_counts = _ann[group_col].value_counts().to_dict()
label_map = {ct: f'{ct} ({class_counts.get(ct, 0)})' for ct in diag_order}
# Display copy with count-annotated labels; keep df_apa (raw names) for the summary.
df_apa_disp = df_apa.rename(index=label_map, columns=label_map)

row_parent = pd.Series({label_map[ct]: group2parent.get(ct, 'merged') for ct in diag_order})
col_parent = row_parent.copy()

row_ha = HeatmapAnnotation(
    Neighborhood=anno_simple(row_parent, colors=parent_colors, legend=True),
    axis=0, verbose=0, label_side='bottom', label_kws=dict(visible=False),
)
col_ha = HeatmapAnnotation(
    Neighborhood=anno_simple(col_parent, colors=parent_colors, legend=False),
    axis=1, verbose=0, label_side='right', label_kws=dict(visible=True),
)

# Tighter figure with larger tick fonts.
plt.figure(figsize=(5.6, 5))
cm = ClusterMapPlotter(
    data=df_apa_disp,
    left_annotation=row_ha,
    top_annotation=col_ha,
    cmap='RdYlBu_r',
    vmin=1.0, vmax=np.nanpercentile(df_apa.values, 95),
    row_cluster=False, col_cluster=False,   # use shared diagonal ordering
    show_rownames=True, show_colnames=False,
    xlabel=f'Diff-loop set ({usecol})',
    ylabel=f'Signal contact matrix Q ({usecol})',
    xlabel_kws={'fontweight': 'bold', 'color': 'black', 'fontsize': 12},
    ylabel_kws={'fontweight': 'bold', 'color': 'black', 'fontsize': 12},
    xlabel_side='top',xticklabels=True, yticklabels=True,
    xticklabels_kws={'labelrotation': 90, 'labelsize': 9},
    yticklabels_kws={'labelsize': 13, 'right': True},
    label='APA strength',
)
# Vertical labels keep the figure narrow without character overlap.
ax_hm = cm.ax_heatmap
for tl in ax_hm.get_xticklabels():
    tl.set_verticalalignment('top')
plt.savefig(os.path.join(figdir, f'{usecol}.cross_celltype_apa_strength.heatmap.pdf'),
            bbox_inches='tight', transparent=True, dpi=300)
plt.show()

# Summary for the manuscript.
diag = np.diag(apa_mat)
off  = apa_mat[~np.eye(N, dtype=bool)]
print(f"""
Summary (paste into manuscript):
  We piled-up each {usecol}-specific differential loop set (columns) on every other
  {usecol}'s (row) imputed contact matrix and computed the centre/background APA
  strength (N={N} x {N}). Diagonal APA strengths (mean {diag.mean():.2f})
  are markedly higher than off-diagonal entries (mean {off.mean():.2f}),
  confirming that the called diff-loops are cell-type-specific and not driven
  by global compaction differences. Rows and columns share the same
  hierarchical ordering, so the high-intensity diagonal directly visualises
  self-recovery while off-diagonal blocks recapitulate the lineage tree
  colour-coded along the row/column annotations.
""")


### (Revision) Per-cell sequencing depth: neuron vs non-neuron

Reviewer control for the cross-Class APA asymmetry: check whether per-cell
`UniqueAlignFinalReads` differs between neuronal and non-neuronal cell types.
Cells are split into Neuron vs Non-neuron by `Neighborhood` (Non-neuron =
`Neighborhood == 'Nonneuron'`), compared with a two-sided Mann-Whitney U test.


In [ ]:
import scipy.stats as ss

# Per-cell sequencing depth (UniqueAlignFinalReads) for the Classes analysed here.
meta_qc = pd.read_csv('/anvil/projects/x-mcb130189/Wubin/BG/clustering/100kb/cell_metadata_with_annotation.tsv',
                      sep='\t', index_col=0)
meta_qc = meta_qc[meta_qc['Class'].isin(group_order)].copy()
meta_qc = meta_qc.dropna(subset=['UniqueAlignFinalReads', 'Neighborhood'])
meta_qc['UniqueAlignFinalReads'] = meta_qc['UniqueAlignFinalReads'].astype(float)
meta_qc['lineage'] = np.where(meta_qc['Neighborhood'] == 'Nonneuron', 'Non-neuron', 'Neuron')

neu = meta_qc.loc[meta_qc.lineage == 'Neuron', 'UniqueAlignFinalReads'].values
non = meta_qc.loc[meta_qc.lineage == 'Non-neuron', 'UniqueAlignFinalReads'].values
U, p = ss.mannwhitneyu(neu, non, alternative='two-sided')
print(f'Neuron:     n={len(neu):>6d}  median={np.median(neu):,.0f}  mean={neu.mean():,.0f}')
print(f'Non-neuron: n={len(non):>6d}  median={np.median(non):,.0f}  mean={non.mean():,.0f}')
print(f'Mann-Whitney U={U:.4g}, two-sided p={p:.3e}')

# Order Classes with non-neuron first, then neuron (grouped by lineage).
nonneu_classes = [c for c in group_order if group2parent.get(c) == 'Nonneuron']
neu_classes    = [c for c in group_order if group2parent.get(c) != 'Nonneuron']
class_order = nonneu_classes + neu_classes
box_pal = {c: parent_colors.get(group2parent.get(c, 'merged'), '#888888') for c in class_order}

fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=200,
                         gridspec_kw={'width_ratios': [1, 3]})

# (a) two-group boxplot with significance annotation.
lin_order = ['Neuron', 'Non-neuron']
lin_pal = {'Neuron': '#c1272d', 'Non-neuron': '#0000a7'}
sns.boxplot(data=meta_qc, x='lineage', y='UniqueAlignFinalReads', order=lin_order,
            hue='lineage', palette=lin_pal, legend=False, showfliers=False,
            width=0.6, ax=axes[0])
axes[0].set_yscale('log')
axes[0].set_ylabel('UniqueAlignFinalReads (log scale)')
axes[0].set_xlabel('')
axes[0].set_title('Per-cell depth by lineage')
ymax = np.percentile(np.concatenate([neu, non]), 98)
axes[0].plot([0, 0, 1, 1], [ymax*1.1, ymax*1.25, ymax*1.25, ymax*1.1], lw=1.0, c='k')
star = 'n.s.' if p >= 0.05 else ('***' if p < 1e-3 else ('**' if p < 1e-2 else '*'))
axes[0].text(0.5, ymax*1.3, f'{star}  (p={p:.1e})', ha='center', va='bottom', fontsize=9)

# (b) per-Class boxplot coloured by Neighborhood.
sns.boxplot(data=meta_qc, x='Class', y='UniqueAlignFinalReads', order=class_order,
            hue='Class', palette=box_pal, legend=False, showfliers=False, ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_ylabel('UniqueAlignFinalReads (log scale)')
axes[1].set_xlabel('')
axes[1].set_title('Per-cell depth by Class (non-neuron | neuron)')
axes[1].set_xticks(range(len(class_order)))
axes[1].set_xticklabels(class_order, rotation=45, ha='right', fontsize=8)
for tl in axes[1].get_xticklabels():
    tl.set_color(parent_colors.get(group2parent.get(tl.get_text(), ''), 'black'))

plt.tight_layout()
plt.savefig(os.path.join(figdir, f'{usecol}.UniqueAlignFinalReads.neuron_vs_nonneuron.boxplot.pdf'),
            bbox_inches='tight', transparent=True, dpi=300)
plt.show()
